# Treino DeepLabV3+ no Kaggle (GPU) — segmentação multiclasse de telhados

Este notebook treina um **DeepLabV3+** (torchvision) com o dataset **roof/chips_multiclass** (5 classes: fundo, águas, claraboia, divisória, laje). O checkpoint gerado é compatível com a API (DeepLabV3 como modelo principal).

**Dataset esperado:** Em `/kaggle/input/` uma pasta **roof/** com `roof/chips_multiclass/images/` e `roof/chips_multiclass/masks/`, ou datasets que permitam montar/criar essa estrutura.

**Saída:** `deeplabv3_roof_multiclass.pt` em `/kaggle/working/models/`. A separação casa a casa e águas por telhado é feita na API com o pipeline atual (DSM, ponto).

In [ ]:
import subprocess
out = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'], capture_output=True, text=True)
USE_GPU = (out.returncode == 0)
if USE_GPU:
    print('GPU:', out.stdout.strip())
else:
    print('Nenhuma GPU. Ativa em Settings → Accelerator → GPU.')

In [ ]:
import os
REPO_URL = "https://github.com/phmotad/roofArea.git"
REPO_DIR = "/kaggle/working/roofArea"

if os.path.isfile(os.path.join(REPO_DIR, "pyproject.toml")) and os.path.isdir(os.path.join(REPO_DIR, "src", "roof_api")):
    print("Projeto já presente em", REPO_DIR)
else:
    if os.path.isdir(REPO_DIR):
        import shutil
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
print("Diretório atual:", os.getcwd())

In [ ]:
subprocess.run(["pip", "install", "-q", "torch", "torchvision", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "pillow", "numpy", "opencv-python-headless", "scikit-image"], check=True)
subprocess.run(["pip", "install", "-q", "-e", "."], check=True)
print("Dependências instaladas.")

Detetar pasta **roof/** ou **chips_multiclass** em `/kaggle/input/`. Se existir roof/ com chips_multiclass, usamos; senão procuramos datasets com nomes típicos.

In [ ]:
import os
KAGGLE_INPUT = "/kaggle/input"

def _has_subdirs(root, *subdirs):
    return all(os.path.isdir(os.path.join(root, d)) for d in subdirs)

def _has_files_in(dirpath, exts=(".png", ".jpg", ".jpeg")):
    if not os.path.isdir(dirpath):
        return False
    try:
        names = os.listdir(dirpath)
        return any(f.lower().endswith(ext) for f in names[:50] for ext in exts)
    except OSError:
        return False

roots_with_name = []
def add_roots(base_path, depth=0):
    if depth > 2:
        return
    try:
        for name in os.listdir(base_path):
            path = os.path.join(base_path, name)
            if os.path.isdir(path):
                roots_with_name.append((path, name.lower()))
                add_roots(path, depth + 1)
    except OSError:
        pass
add_roots(KAGGLE_INPUT)

CHIPS_MC_PATH = ""
ROOF_DIR = ""
BUILD_ROOF = False

for root, _ in roots_with_name:
    if _has_subdirs(root, "chips", "chips_multiclass") and _has_files_in(os.path.join(root, "chips_multiclass", "images")):
        ROOF_DIR = root
        CHIPS_MC_PATH = os.path.join(ROOF_DIR, "chips_multiclass")
        break

if not CHIPS_MC_PATH:
    for root, name in roots_with_name:
        cand = os.path.join(root, "chips_multiclass") if os.path.isdir(os.path.join(root, "chips_multiclass")) else root
        if _has_subdirs(cand, "images", "masks") and _has_files_in(os.path.join(cand, "images")):
            CHIPS_MC_PATH = cand
            break

if not CHIPS_MC_PATH:
    for top in os.listdir(KAGGLE_INPUT):
        base = os.path.join(KAGGLE_INPUT, top)
        p = os.path.join(base, "roof-chips-multiclass", "chips_multiclass")
        if os.path.isdir(p) and _has_subdirs(p, "images", "masks"):
            CHIPS_MC_PATH = p
            break

if not CHIPS_MC_PATH:
    raise SystemExit("Nenhum dataset chips_multiclass em /kaggle/input/. Anexa um dataset com roof/chips_multiclass ou roof-chips-multiclass.")

print("Chips multiclass:", CHIPS_MC_PATH)

In [ ]:
import sys
import torch
from pathlib import Path
from torch.utils.data import DataLoader, random_split

sys.path.insert(0, REPO_DIR)
from roof_api.segmentation.dataset import RoofDataset

NUM_CLASSES = 5
SIZE = (512, 512)
BATCH_SIZE = 4
VAL_RATIO = 0.2
device = torch.device("cuda" if USE_GPU and torch.cuda.is_available() else "cpu")
print("Device:", device)

images_dir = Path(CHIPS_MC_PATH) / "images"
masks_dir = Path(CHIPS_MC_PATH) / "masks"
full_ds = RoofDataset(images_dir, masks_dir, size=SIZE, augment=True, num_classes=NUM_CLASSES)
n = len(full_ds)
n_val = max(1, int(n * VAL_RATIO))
n_train = n - n_val
train_ds, val_ds = random_split(full_ds, [n_train, n_val], generator=torch.Generator().manual_seed(42))
print("Train:", n_train, "Val:", n_val)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=(device.type == "cuda"))
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

In [ ]:
from torchvision.models.segmentation import deeplabv3_resnet50

model = deeplabv3_resnet50(num_classes=NUM_CLASSES, weights=None, weights_backbone=None)
model = model.to(device)
criterion = torch.nn.CrossEntropyLoss(ignore_index=255)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
print("Modelo DeepLabV3 ResNet50, num_classes=", NUM_CLASSES)

In [ ]:
EPOCHS = 30
OUTPUT_PATH = "/kaggle/working/models/deeplabv3_roof_multiclass.pt"
os.makedirs("/kaggle/working/models", exist_ok=True)

best_val_loss = float("inf")

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        out = model(x)
        logits = out["out"]
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * x.size(0)
    train_loss /= n_train

    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            logits = out["out"]
            val_loss += criterion(logits, y).item() * x.size(0)
    val_loss /= n_val

    print(f"Epoch {epoch} | train loss={train_loss:.4f} | val loss={val_loss:.4f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save({"model": model.state_dict(), "num_classes": NUM_CLASSES}, OUTPUT_PATH)
        print(f"  -> Guardado melhor modelo em {OUTPUT_PATH}")

print("Treino concluído. Melhor modelo:", OUTPUT_PATH)

In [ ]:
import os
for f in sorted(os.listdir("/kaggle/working/models")):
    path = os.path.join("/kaggle/working/models", f)
    print(f, "—", os.path.getsize(path) // 1024, "KB")
print("\nDescarrega deeplabv3_roof_multiclass.pt e coloca em models/ no teu projeto. Configura SEGMENTATION_MODEL_PATH=./models/deeplabv3_roof_multiclass.pt")